In [ ]:
%pip install librosa numpy scipy matplotlib
%pip install soundfile

In [ ]:
#aiffファイルをwavファイルに変換
import soundfile as sf

file_aiff = "Trumpet_A4.aiff"  # ダウンロードしたAIFFのファイル名
data, samplerate = sf.read(file_aiff)

# WAVファイルとして書き出す
sf.write("trumpet_real_A4.wav", data, samplerate)

print("WAVファイルへの変換が完了しました！左のファイル一覧を確認してください．")

In [ ]:
#エンベロープの相関係数
import numpy as np
from scipy.io import wavfile
from scipy.stats import pearsonr
import matplotlib.pyplot as plt

def evaluate_envelope(file_real, file_make, window_ms=30):
    print("音声ファイルを読み込み中...")
    
    # scipy.io.wavfile を使って読み込む
    sr_real, y_real = wavfile.read(file_real)
    sr_make, y_make = wavfile.read(file_make)
    
    # 小数の配列に変換
    if y_real.dtype == np.int16:
        y_real = y_real.astype(np.float32) / 32768.0
    if y_make.dtype == np.int16:
        y_make = y_make.astype(np.float32) / 32768.0
        
    # ステレオならモノラルに強制変換
    if y_real.ndim > 1:
        y_real = y_real[:, 0]
    if y_make.ndim > 1:
        y_make = y_make[:, 0]
        
    if sr_real != sr_make:
        print(f"【警告】サンプリング周波数が一致していません（実機:{sr_real}Hz, 合成:{sr_make}Hz）")
    
    frame_length = int(sr_real * (window_ms / 1000.0))
    
    def get_rms_envelope(y, frame_len):
        rms_list = []
        for i in range(0, len(y), frame_len):
            frame = y[i : i + frame_len]
            rms = np.sqrt(np.mean(frame**2) + 1e-10)
            rms_list.append(rms)
        return np.array(rms_list)

    rms_real = get_rms_envelope(y_real, frame_length)
    rms_make = get_rms_envelope(y_make, frame_length)
    

    max_len = max(len(rms_real), len(rms_make))
    
    # np.pad を使って、足りない分だけ 0 を後ろに足す
    rms_real_padded = np.pad(rms_real, (0, max_len - len(rms_real)), 'constant')
    rms_make_padded = np.pad(rms_make, (0, max_len - len(rms_make)), 'constant')
    
    # 正規化
    if np.max(rms_real_padded) > 0:
        rms_real_norm = rms_real_padded / np.max(rms_real_padded)
    else:
        rms_real_norm = rms_real_padded
        
    if np.max(rms_make_padded) > 0:
        rms_make_norm = rms_make_padded / np.max(rms_make_padded)
    else:
        rms_make_norm = rms_make_padded
    
    # 長さを揃えた状態でピアソン相関を計算
    correlation, _ = pearsonr(rms_real_norm, rms_make_norm)
    
    # グラフの描画
    time_axis = np.arange(max_len) * (window_ms / 1000.0)
    
    plt.figure(figsize=(10, 4))
    plt.plot(time_axis, rms_real_norm, label="Real", color="blue", alpha=0.7)
    plt.plot(time_axis, rms_make_norm, label="make", color="orange", alpha=0.7, linestyle="--")
    plt.title(f"Envelope Comparison (Pearson Correlation: {correlation:.4f})")
    plt.xlabel("Time [seconds]")
    plt.ylabel("Normalized RMS")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    
    return correlation

# ==========================================
# 実行部分
# ==========================================
file_A = "trumpet_real_A4.wav"   # 実機音源のファイル名
file_B = "trumpet_make_A4.wav"  # 合成音源のファイル名

# 関数の実行
corr = evaluate_envelope(file_A, file_B, window_ms=30)

print("\n=== 評価結果 ===")
print(f"ピアソンの積率相関係数: {corr:.4f}")

if corr >= 0.9:
    print("判定: [成功] 相関係数 0.9 以上を達成しています！")
else:
    print("判定: [改善が必要] 相関係数が 0.9 未満です．")

plt.show()

In [ ]:
#倍音構成の各倍音の大きさ
import numpy as np
from scipy.io import wavfile
from scipy.signal import welch
import matplotlib.pyplot as plt

def compare_harmonics_table_and_graph(file_real, file_make, max_freq=15000):
    print("音声ファイルを読み込み、高次倍音を抽出中...\n")
    
    # 音源の読み込み
    sr_real, y_real = wavfile.read(file_real)
    sr_make, y_make = wavfile.read(file_make)
    
    if y_real.dtype == np.int16:
        y_real = y_real.astype(np.float32) / 32768.0
    if y_make.dtype == np.int16:
        y_make = y_make.astype(np.float32) / 32768.0
    
    if y_real.ndim > 1: y_real = y_real[:, 0]
    if y_make.ndim > 1: y_make = y_make[:, 0]

    n_window = 16384 
    f_real, pxx_real = welch(y_real, sr_real, nperseg=n_window)
    f_make, pxx_make = welch(y_make, sr_make, nperseg=n_window)
    
    amp_real = np.sqrt(pxx_real)
    amp_make = np.sqrt(pxx_make)
    
    amp_real_norm = amp_real / np.max(amp_real)
    amp_make_norm = amp_make / np.max(amp_make)
    
    # 基音（A4）の探索
    search_idx = np.where((f_real >= 400) & (f_real <= 500))[0]
    if len(search_idx) > 0:
        f0_idx = search_idx[np.argmax(amp_real_norm[search_idx])]
        f0 = f_real[f0_idx]
    else:
        f0_idx = np.argmax(amp_real_norm)
        f0 = f_real[f0_idx]

    print(f"基準となる基音: {f0:.1f} Hz\n")
    
    # ==========================================
    # 1. 表形式で出力
    # ==========================================
    print("-" * 55)
    print(f"{'倍音':<5} | {'周波数 (Hz)':<12} | {'Real':<12} | {'make':<12}")
    print("-" * 55)
    
    i = 1
    harmonics_real = []
    
    while True:
        target_f = f0 * i
        if target_f > max_freq:
            break
            
        idx_r = np.where(np.abs(f_real - target_f) < 50)[0]
        val_r = np.max(amp_real_norm[idx_r]) if len(idx_r) > 0 else 0.0
        
        idx_m = np.where(np.abs(f_make - target_f) < 50)[0]
        val_m = np.max(amp_make_norm[idx_m]) if len(idx_m) > 0 else 0.0
        
        harmonics_real.append(val_r)
        
        print(f" {i:<4} | {target_f:<12.1f} | {val_r:<12.3f} | {val_m:<12.3f}")
        i += 1
        
    print("-" * 55)

    print("\n=== Processing配列コピペ用（実機の全倍音） ===")
    float_array_str = ", ".join([f"{amp:.3f}f" for amp in harmonics_real])
    print(f"new float[] {{ {float_array_str} }}")
    print("============================================\n")

    # ==========================================
    # 2. グラフの描画
    # ==========================================
    plt.figure(figsize=(12, 5))
    
    # max_freq (今回は15000Hz) までのデータだけを切り出す
    valid_idx_r = f_real <= max_freq
    valid_idx_m = f_make <= max_freq
    
    # Realの描画
    plt.plot(f_real[valid_idx_r], amp_real_norm[valid_idx_r], label="Real", color="blue", alpha=0.7)
    
    # makeの描画
    plt.plot(f_make[valid_idx_m], amp_make_norm[valid_idx_m], label="make", color="orange", alpha=0.8, linestyle="--")
    
    plt.title("Harmonics Comparison (Real vs make)")
    plt.xlabel("Frequency [Hz]")
    plt.ylabel("Relative Amplitude [0.0 - 1.0]")
    plt.xlim(0, max_freq)
    plt.ylim(0, 1.05) 
    
    plt.legend()
    plt.grid(True, alpha=0.5)
    plt.tight_layout()
    
    # グラフを表示する
    plt.show()

# ==========================================
# 実行部分
# ==========================================
file_A = "trumpet_real_A4.wav"   
file_B = "trumpet_make_A4.wav"   

# 15000Hz までのデータを表とグラフで比較
compare_harmonics_table_and_graph(file_A, file_B, max_freq=15000)

In [ ]:
import numpy as np
from scipy.io import wavfile
from scipy.signal import welch
import matplotlib.pyplot as plt

def visualize_rmse(file_real, file_make, max_freq=30000):
    print("音声ファイルを読み込み、RMSEを計算中...")
    
    sr_real, y_real = wavfile.read(file_real)
    sr_make, y_make = wavfile.read(file_make)
    
    if y_real.dtype == np.int16:
        y_real = y_real.astype(np.float32) / 32768.0
    if y_make.dtype == np.int16:
        y_make = y_make.astype(np.float32) / 32768.0
        
    if y_real.ndim > 1: y_real = y_real[:, 0]
    if y_make.ndim > 1: y_make = y_make[:, 0]
    
    n_window = 4096
    f_real, pxx_real = welch(y_real, sr_real, nperseg=n_window)
    f_make, pxx_make = welch(y_make, sr_make, nperseg=n_window)
    
    valid_idx = f_real <= max_freq
    freqs = f_real[valid_idx]
    
    pxx_real = pxx_real[valid_idx]
    pxx_make = pxx_make[valid_idx]
    
    # 頂点が0dBになるように正規化
    pxx_real_norm = pxx_real / np.max(pxx_real)
    pxx_make_norm = pxx_make / np.max(pxx_make)
    
    # dBに変換
    db_real = 10 * np.log10(pxx_real_norm + 1e-10)
    db_make = 10 * np.log10(pxx_make_norm + 1e-10)
    
    # RMSE（二乗平均平方根誤差）を計算
    rmse_val = np.sqrt(np.mean((db_real - db_make)**2))
    
    # ==========================================
    # コンソールへの出力
    # ==========================================
    print("\n" + "="*40)
    print(f" 全体的なRMSE (誤差): {rmse_val:.4f} dB")
    print("="*40 + "\n")
    
    # ==========================================
    # グラフ描画
    # ==========================================
    plt.figure(figsize=(12, 6))
    
    # 波形の比較のみを描画
    plt.plot(freqs, db_real, label="Real", color="blue", alpha=0.8)
    plt.plot(freqs, db_make, label="make", color="orange", alpha=0.8, linestyle="--")
    
    # タイトルにRMSEの数値を表示
    plt.title(f"Spectrum Comparison (Overall RMSE: {rmse_val:.4f} dB)")
    plt.xlabel("Frequency [Hz]")
    plt.ylabel("Magnitude [dB]")
    plt.legend()
    plt.grid(True, alpha=0.5)
    
    plt.tight_layout()
    plt.show()

# ==========================================
# 実行部分
# ==========================================
file_A = "trumpet_real_A4.wav"
file_B = "trumpet_make_A4.wav"

visualize_rmse(file_A, file_B, max_freq=30000)